# TransfIA — Fine-tuning LoRA d'un petit modèle (F10)

**À exécuter sur Google Colab ou Kaggle avec un GPU gratuit** (Exécution → Modifier le type d'exécution → GPU T4).

Ce que fait ce notebook :
1. charge un petit modèle (1,5 Md de paramètres) **en 4 bits** (QLoRA) ;
2. ajoute des **adaptateurs LoRA** (~1 % des paramètres entraînables) ;
3. l'entraîne (SFT) sur `train.jsonl` produit par `finetuning/build_dataset.py` ;
4. surveille la **perte de validation** (sur-apprentissage ?) ;
5. exporte en **GGUF 4 bits** pour l'utiliser ensuite **sur ton PC, sans GPU**, avec Ollama.

⚠️ Les API d'Unsloth et de TRL évoluent vite : en cas d'erreur, compare avec leur documentation.

In [ ]:
!pip install -q unsloth
# Téléverse train.jsonl et val.jsonl dans le panneau "Fichiers" de Colab (à gauche) avant de continuer.

## 1. Charger le modèle en 4 bits et ajouter LoRA

In [ ]:
from unsloth import FastLanguageModel

MODEL = "unsloth/Qwen2.5-1.5B-Instruct"   # même famille que le modèle local : comparaison équitable
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL, max_seq_length=2048, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                 # rang de la correction ΔW = B·A
    lora_alpha=16,        # facteur d'échelle α / r
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model.print_trainable_parameters()

## 2. Données : format chat → texte avec le template du modèle

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

def to_text(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

ds = ds.map(to_text)
print(ds["train"][0]["text"][:800])

## 3. Entraînement supervisé (SFT)

In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer

# Petit jeu (~36 exemples) : 36 / (2 x 4) ≈ 5 pas par époque. On journalise CHAQUE pas et on évalue à chaque
# époque (avec eval_steps=20, aucune évaluation n'aurait eu lieu : 10 pas au total).
cfg = dict(dataset_text_field="text", per_device_train_batch_size=2, gradient_accumulation_steps=4,
           num_train_epochs=3, learning_rate=2e-4, warmup_steps=2, logging_steps=1,
           eval_strategy="epoch", output_dir="outputs", seed=42, report_to="none")
# TRL a renommé des paramètres selon les versions : on s'adapte à celle installée par Colab.
cfg_params = inspect.signature(SFTConfig.__init__).parameters
cfg["max_length" if "max_length" in cfg_params and "max_seq_length" not in cfg_params else "max_seq_length"] = 2048

kwargs = dict(model=model, train_dataset=ds["train"], eval_dataset=ds["validation"], args=SFTConfig(**cfg))
try:
    trainer = SFTTrainer(processing_class=tokenizer, **kwargs)      # versions récentes de TRL
except TypeError:
    trainer = SFTTrainer(tokenizer=tokenizer, **kwargs)              # versions plus anciennes
stats = trainer.train()


## 4. Lire les courbes

- La perte d'**entraînement** (`loss`) baisse toujours.
- La perte de **validation** (`eval_loss`, une valeur par époque) doit baisser aussi.
- Si `eval_loss` remonte pendant que `loss` baisse : **sur-apprentissage** → moins d'époques ou plus de données.
- Avec ~36 exemples, les courbes sont bruitées : c'est une démonstration du mécanisme, pas un modèle de production.


In [ ]:
import pandas as pd
logs = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["step", "epoch", "loss", "eval_loss"] if c in logs]
print(logs[cols].dropna(how="all", subset=[c for c in ["loss", "eval_loss"] if c in logs]).to_string())


## 5. La mesure : modèle de BASE vs modèle FINE-TUNÉ, sur les exemples de validation

On demande à chacun un QCM pour des passages qu'il n'a **pas** vus à l'entraînement, **sans** contrainte de format
(pas de JSON schema), avec le MÊME prompt système qu'à l'entraînement. On mesure :
- **JSON valide du premier coup** (le fine-tuning apprend ce FORMAT) ;
- **QCM conforme** : 4 choix distincts, une bonne réponse entre 0 et 3.

À retenir : le fine-tuning apprend un **comportement** (format, style), pas des **connaissances**.


In [ ]:
import json

FastLanguageModel.for_inference(model)
val = [json.loads(line) for line in open("val.jsonl", encoding="utf-8")]

def generate(messages):
    inputs = tokenizer.apply_chat_template(messages[:2], add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=700, max_length=None, temperature=0.2, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

def score(text):
    try:
        d = json.loads(text.strip().removeprefix("```json").removesuffix("```").strip())
    except Exception:
        return False, False                      # JSON invalide
    qs = d.get("questions") if isinstance(d, dict) else None
    if not qs:
        return True, False
    try:
        ok = all(isinstance(q, dict) and len(set(q.get("choix", []))) == 4
                 and isinstance(q.get("bonne_reponse"), int) and 0 <= q["bonne_reponse"] <= 3
                 for q in qs)
    except Exception:
        ok = False                               # mauvais type (vu : texte au lieu d'un numéro) = non conforme
    return True, ok

res = {"base": [], "fine-tuné": []}
for ex in val:
    with model.disable_adapter():                   # le MÊME modèle, sans les adaptateurs LoRA = modèle de base
        res["base"].append(score(generate(ex["messages"])))
    res["fine-tuné"].append(score(generate(ex["messages"])))

for name, r in res.items():
    print(f"{name:10s} JSON valide : {sum(a for a, _ in r)}/{len(r)}  —  QCM conforme : {sum(b for _, b in r)}/{len(r)}")
print("\nExemple (fine-tuné) :\n", generate(val[0]["messages"])[:600])


## 6. Export GGUF 4 bits (pour Ollama sur ton PC)

La compilation de l'outil d'export prend plusieurs minutes. Le nom exact du fichier produit dépend de la version
d'Unsloth : la cellule suivante le cherche et l'affiche. Télécharge-le (panneau Fichiers, clic droit → Télécharger).


In [ ]:
model.save_pretrained_gguf("transfia-qcm", tokenizer, quantization_method="q4_k_m")
import glob, os
for f in glob.glob("**/*.gguf", recursive=True):
    print(f, f"{os.path.getsize(f) / 1e9:.2f} Go")
# Sur ton PC : renomme le fichier téléchargé en transfia-qcm.Q4_K_M.gguf, place-le dans finetuning/,
# puis : ollama create transfia-qcm -f finetuning/Modelfile
